# NYSE and Airlines Data Analysis

This notebook demonstrates DataFrame operations on NYSE stock data and Airlines dataset.

In [ ]:
# Install and setup Java (for Google Colab)
import os

def install_java():
    !apt-get install -y openjdk-8-jdk-headless -qq > /dev/null
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
    !java -version

install_java()

In [ ]:
# Install PySpark
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StringType, IntegerType, DoubleType, LongType
from pyspark.sql.functions import col, sum, avg, round as spark_round, desc

spark = SparkSession.builder \
    .appName('NYSE and Airlines Analysis') \
    .getOrCreate()

print(f"Spark Version: {spark.version}")

## Part 1: NYSE Stock Data Analysis

### Define Schema for NYSE Data

In [ ]:
# Define explicit schema for NYSE data
schema_nyse = StructType() \
    .add("exchange_name", StringType(), True) \
    .add("stock_id", StringType(), True) \
    .add("stock_dt", StringType(), True) \
    .add("open", DoubleType(), True) \
    .add("high", DoubleType(), True) \
    .add("low", DoubleType(), True) \
    .add("close", DoubleType(), True) \
    .add("volume", LongType(), True) \
    .add("adj_close", DoubleType(), True)

print(schema_nyse)

In [ ]:
# Read NYSE data with schema
# Update the path to your NYSE.csv location
nyse_df = spark.read \
    .format("csv") \
    .option("header", "False") \
    .schema(schema_nyse) \
    .load("/content/sample_data/NYSE.csv")

nyse_df.printSchema()
nyse_df.show(5)

### Register as Temp Table and Run SQL Queries

In [ ]:
# Register as temporary view
nyse_df.createOrReplaceTempView("nyse")

# Alternative method
# nyse_df.registerTempTable("nyse")

In [ ]:
# Calculate total volume per stock
df_stock_vol = spark.sql("""
    SELECT stock_id, SUM(volume) as total_volume 
    FROM nyse 
    GROUP BY stock_id
    ORDER BY total_volume DESC
""")

df_stock_vol.show(10)

### Check and Adjust Partitions

In [ ]:
# Check number of partitions
print(f"Number of partitions: {df_stock_vol.rdd.getNumPartitions()}")

# Coalesce to 1 partition for output
df_stock_vol_single = df_stock_vol.coalesce(1)
print(f"After coalesce: {df_stock_vol_single.rdd.getNumPartitions()}")

In [ ]:
# Adjust shuffle partitions configuration
spark.conf.set("spark.sql.shuffle.partitions", 100)

# Re-run query to see effect
df_stock_vol_100 = spark.sql("""
    SELECT stock_id, SUM(volume) as total_volume 
    FROM nyse 
    GROUP BY stock_id
""")

print(f"Partitions with shuffle.partitions=100: {df_stock_vol_100.rdd.getNumPartitions()}")

### Save Results

In [ ]:
# Save to CSV
output_path = "/tmp/nyse_stock_volume"
!rm -rf {output_path}

df_stock_vol_single.write \
    .mode("overwrite") \
    .csv(output_path)

print(f"Results saved to {output_path}")

## Part 2: Airlines Data Analysis

In [ ]:
# Read airlines data (with schema inference)
airlines_df = spark.read \
    .format("csv") \
    .option("header", "True") \
    .option("inferSchema", "True") \
    .load("/content/sample_data/airlines.csv")

airlines_df.printSchema()
airlines_df.show(5)

In [ ]:
# Register as temporary view
airlines_df.createOrReplaceTempView("airlines")

### Year-wise Revenue Analysis

In [ ]:
# Calculate year-wise revenue (in millions)
yr_wise_rev = spark.sql("""
    SELECT 
        year, 
        ROUND(SUM(Avg_rev_per_seat * booked_seats) / 1000000, 2) as total_in_mill 
    FROM airlines 
    GROUP BY year 
    ORDER BY total_in_mill DESC
""")

print("Year-wise Revenue (in Millions):")
yr_wise_rev.show()

### Year-wise Passenger Count

In [ ]:
# Calculate year-wise total passengers
yr_wise_psx = spark.sql("""
    SELECT 
        year, 
        SUM(booked_seats) as total_psx 
    FROM airlines 
    GROUP BY year 
    ORDER BY total_psx DESC
""")

print("Year-wise Passenger Count:")
yr_wise_psx.show()

### Combined Analysis

In [ ]:
# Combined year-wise analysis
combined_analysis = spark.sql("""
    SELECT 
        year,
        ROUND(SUM(Avg_rev_per_seat * booked_seats) / 1000000, 2) as revenue_millions,
        SUM(booked_seats) as total_passengers,
        ROUND(AVG(Avg_rev_per_seat), 2) as avg_revenue_per_seat
    FROM airlines 
    GROUP BY year 
    ORDER BY year
""")

print("Combined Year-wise Analysis:")
combined_analysis.show()

### Quarter-wise Analysis

In [ ]:
# Year and Quarter wise analysis
quarter_analysis = spark.sql("""
    SELECT 
        year,
        quarter,
        ROUND(SUM(Avg_rev_per_seat * booked_seats) / 1000000, 2) as revenue_millions,
        SUM(booked_seats) as total_passengers
    FROM airlines 
    GROUP BY year, quarter
    ORDER BY year, quarter
""")

print("Year and Quarter-wise Analysis:")
quarter_analysis.show(20)

In [ ]:
# Stop Spark Session
spark.stop()